# Exercice 08 : Nettoyer et justifier

Objectif : produire un fichier `data/titanic_propre.csv` propre, avec un journal qui explique chaque décision de nettoyage.

In [1]:
import os

import seaborn as sns
import pandas as pd

titanic = sns.load_dataset("titanic")
print(titanic.shape)

(891, 15)


## 1. Les valeurs manquantes

In [2]:
print(titanic.isna().sum())
print(titanic.isna().mean().round(3))

survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64
survived       0.000
pclass         0.000
sex            0.000
age            0.199
sibsp          0.000
parch          0.000
fare           0.000
embarked       0.002
class          0.000
who            0.000
adult_male     0.000
deck           0.772
embark_town    0.002
alive          0.000
alone          0.000
dtype: float64


## 2. Les doublons

On les examine **avant** d'imputer l'âge : l'imputation donnerait le même âge à des passagers différents et créerait de faux doublons.

In [3]:
print(titanic.duplicated().sum())
print(titanic[titanic.duplicated(keep=False)].head(10))

107
    survived  pclass     sex   age  sibsp  parch     fare embarked  class  \
4          0       3    male  35.0      0      0   8.0500        S  Third   
26         0       3    male   NaN      0      0   7.2250        C  Third   
28         1       3  female   NaN      0      0   7.8792        Q  Third   
29         0       3    male   NaN      0      0   7.8958        S  Third   
32         1       3  female   NaN      0      0   7.7500        Q  Third   
37         0       3    male  21.0      0      0   8.0500        S  Third   
42         0       3    male   NaN      0      0   7.8958        C  Third   
45         0       3    male   NaN      0      0   8.0500        S  Third   
46         0       3    male   NaN      1      0  15.5000        Q  Third   
47         1       3  female   NaN      0      0   7.7500        Q  Third   

      who  adult_male deck  embark_town alive  alone  
4     man        True  NaN  Southampton    no   True  
26    man        True  NaN    Cherbour

| Quoi | Pourquoi | Lignes |
|---|---|---|
| Doublons **conservés** | Le jeu n'a pas d'identifiant ni de nom : deux lignes identiques (souvent des hommes de 3e classe, même tarif, âge inconnu) peuvent être deux vraies personnes. Les supprimer fausserait les effectifs. | 107 |

## 3. Imputer l'âge par la médiane de la classe et du sexe

In [4]:
mediane = titanic.groupby(["pclass", "sex"])["age"].transform("median")
print(titanic.groupby(["pclass", "sex"])["age"].median())
titanic["age"] = titanic["age"].fillna(mediane)
print(titanic["age"].isna().sum())

pclass  sex   
1       female    35.0
        male      40.0
2       female    28.0
        male      30.0
3       female    21.5
        male      25.0
Name: age, dtype: float64
0


| Quoi | Pourquoi | Lignes |
|---|---|---|
| `age` imputé | 20 % de manquants : trop pour supprimer les lignes. L'âge dépend nettement de la classe et du sexe (de 21,5 ans à 40 ans selon le groupe), la médiane par groupe est donc plus juste qu'une médiane globale. | 177 |

## 4. Port d'embarquement manquant

In [5]:
print(titanic[titanic["embark_town"].isna()])
print(titanic["embark_town"].value_counts())

     survived  pclass     sex   age  sibsp  parch  fare embarked  class  \
61          1       1  female  38.0      0      0  80.0      NaN  First   
829         1       1  female  62.0      0      0  80.0      NaN  First   

       who  adult_male deck embark_town alive  alone  
61   woman       False    B         NaN   yes   True  
829  woman       False    B         NaN   yes   True  
embark_town
Southampton    644
Cherbourg      168
Queenstown      77
Name: count, dtype: int64


In [6]:
port_majoritaire = titanic["embark_town"].mode()[0]
titanic["embark_town"] = titanic["embark_town"].fillna(port_majoritaire)
print(titanic["embark_town"].isna().sum())

0


| Quoi | Pourquoi | Lignes |
|---|---|---|
| `embark_town` imputé par Southampton | Seulement 2 passagères concernées. Southampton est le port de 72 % des passagers, on garde ainsi ces 2 lignes au lieu de les perdre. | 2 |

## 5. La colonne `deck`

In [7]:
print(f"Part de deck manquant : {titanic['deck'].isna().mean():.1%}")
titanic = titanic.drop(columns=["deck"])

Part de deck manquant : 77.2%


| Quoi | Pourquoi | Lignes |
|---|---|---|
| `deck` supprimée | 77 % de manquants (688 sur 891) : impossible d'imputer sans inventer l'essentiel de la colonne. | 891 |

## 6. Colonnes redondantes

In [8]:
print((titanic["class"].cat.codes + 1 == titanic["pclass"]).all())
print(pd.crosstab(titanic["embarked"], titanic["embark_town"]))
print(pd.crosstab(titanic["alive"], titanic["survived"]))

True
embark_town  Cherbourg  Queenstown  Southampton
embarked                                       
C                  168           0            0
Q                    0          77            0
S                    0           0          644
survived    0    1
alive             
no        549    0
yes         0  342


In [9]:
titanic = titanic.drop(columns=["alive", "class", "embarked"])
print(titanic.columns.tolist())

['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'who', 'adult_male', 'embark_town', 'alone']


| Quoi | Pourquoi | Lignes |
|---|---|---|
| `alive`, `class`, `embarked` supprimées | Copies exactes de `survived`, `pclass` et `embark_town` (vérifié ci-dessus). Les garder crée un risque d'incohérence. | 891 |

## 7. Les types

In [10]:
print(titanic.dtypes)

survived         int64
pclass           int64
sex             object
age            float64
sibsp            int64
parch            int64
fare           float64
who             object
adult_male        bool
embark_town     object
alone             bool
dtype: object


In [11]:
titanic["sex"] = titanic["sex"].astype("category")
titanic["embark_town"] = titanic["embark_town"].astype("category")
titanic["who"] = titanic["who"].astype("category")
print(titanic.dtypes)

survived          int64
pclass            int64
sex            category
age             float64
sibsp             int64
parch             int64
fare            float64
who            category
adult_male         bool
embark_town    category
alone              bool
dtype: object


| Quoi | Pourquoi | Lignes |
|---|---|---|
| `sex`, `embark_town`, `who` en `category` | Ce sont des catégories avec peu de valeurs possibles, pas du texte libre. `survived` reste en 0/1 : sa moyenne donne directement le taux de survie. Les autres types (`age`, `fare` en float, `pclass` en int) étaient déjà corrects. | 891 |

## 8. Les billets hors bornes

In [12]:
print(titanic["fare"].describe())

count    891.000000
mean      32.204208
std       49.693429
min        0.000000
25%        7.910400
50%       14.454200
75%       31.000000
max      512.329200
Name: fare, dtype: float64


In [13]:
q1 = titanic["fare"].quantile(0.25)
q3 = titanic["fare"].quantile(0.75)
ecart = q3 - q1
borne_haute = q3 + 1.5 * ecart
tres_chers = titanic[titanic["fare"] > borne_haute]
print(f"Borne haute : {borne_haute:.2f}")
print(f"Billets au-dessus : {len(tres_chers)}")
print(tres_chers["pclass"].value_counts())
print(f"Billets à 0 : {(titanic['fare'] == 0).sum()}")

Borne haute : 65.63
Billets au-dessus : 116
pclass
1    104
3      7
2      5
Name: count, dtype: int64
Billets à 0 : 15


| Quoi | Pourquoi | Lignes |
|---|---|---|
| Billets chers **conservés** | 116 billets dépassent la borne de 65,63 £, dont 104 en 1re classe : ce sont des tarifs rares mais réels (le maximum, 512 £, correspond à des suites). Aberrant ne veut pas dire faux. | 116 |
| Billets à 0 **conservés** | 15 passagers sans tarif : plausiblement des invités ou des membres de l'équipage voyageant gratuitement, pas une erreur de saisie évidente. | 15 |

## 9. Enregistrement

In [14]:
os.makedirs("data", exist_ok=True)
titanic.to_csv("data/titanic_propre.csv", index=False)
print(titanic.shape)
print(titanic.isna().sum().sum())

(891, 11)
0


## Journal de nettoyage

| Quoi | Pourquoi | Lignes |
|---|---|---|
| Doublons conservés | Pas d'identifiant : peuvent être de vraies personnes | 107 |
| `age` imputé | Médiane par classe et sexe, 20 % de manquants | 177 |
| `embark_town` imputé | Port majoritaire (Southampton), 2 lignes seulement | 2 |
| `deck` supprimée | 77 % de manquants | 891 |
| `alive`, `class`, `embarked` supprimées | Redondantes avec `survived`, `pclass`, `embark_town` | 891 |
| `sex`, `embark_town`, `who` en category | Variables catégorielles | 891 |
| Billets chers conservés | Tarifs rares mais réels, surtout en 1re classe | 116 |
| Billets à 0 conservés | Pas d'erreur évidente | 15 |

Résultat : 891 lignes, 11 colonnes, aucune valeur manquante.